# Занятие 35. Решающее дерево

Решающее дерево принимает решение последовательностью вопросов «да / нет». Оно легко строит нелинейные правила и часто понятно человеку, но без ограничений быстро переобучается.


## 1. Устройство дерева

- **Корень** — первый вопрос.
- **Внутренний узел** — следующий вопрос.
- **Ветвь** — вариант ответа.
- **Лист** — итоговый прогноз.

Например: «подготовка ≤ 4 часов?» → «сон ≤ 6 часов?» → прогноз класса.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

X=pd.DataFrame({'подготовка':[1,2,3,4,5,6,7,8,4,6], 'сон':[5,8,6,7,5,8,6,9,9,4]})
y=np.array([0,0,0,0,1,1,1,1,1,0])
tree=DecisionTreeClassifier(max_depth=3,random_state=42).fit(X,y)
plt.figure(figsize=(12,6)); plot_tree(tree,feature_names=X.columns,class_names=['не сдал','сдал'],filled=True); plt.show()


## 2. Рекурсивное построение

После первого вопроса алгоритм рассматривает левую и правую группы как новые маленькие задачи и снова ищет лучший вопрос. Повторение одного правила внутри полученных частей называется рекурсией.

Построение прекращается, когда узел достаточно чист, слишком мал или достигнуто ограничение глубины.


## 3. Как выбирается вопрос

Алгоритм перебирает признаки и пороги. Хороший вопрос разделяет объекты так, чтобы в дочерних узлах классы стали более однородными.

Один из критериев — **Gini impurity**:

$$Gini=1-\sum_k p_k^2.$$

Gini равен 0, если в узле один класс. Чем сильнее смешаны классы, тем он больше.


In [ ]:
def gini(labels):
 counts=np.bincount(labels); p=counts/counts.sum(); return 1-np.sum(p**2)

for labels in [np.array([1,1,1,1]),np.array([0,0,1,1]),np.array([0,0,0,1])]:
 print(labels,'Gini =',round(gini(labels),3))


## 4. Качество разбиения

После вопроса считают взвешенную неоднородность дочерних узлов:

$$Gini_{split}=\frac{n_L}{n}Gini_L+\frac{n_R}{n}Gini_R.$$

Выбирают разбиение с самым большим уменьшением impurity родительского узла.

**Энтропия (*)** — другой критерий со схожей идеей. Для первого знакомства достаточно Gini: на практике результаты часто близки.


In [ ]:
parent=np.array([0,0,0,1,1,1])
splits={
 'A: классы разделены':(np.array([0,0,0]),np.array([1,1,1])),
 'B: классы смешаны':(np.array([0,1,0]),np.array([1,0,1]))
}
for name,(left,right) in splits.items():
 weighted=(len(left)*gini(left)+len(right)*gini(right))/len(parent)
 gain=gini(parent)-weighted
 print(name,'взвешенный Gini =',round(weighted,3),'улучшение =',round(gain,3))


## 5. Энтропия *

Энтропия равна нулю в чистом узле и растёт при смешивании классов, как Gini. Формула использует логарифм, но смысл тот же: хороший вопрос уменьшает неопределённость.

Выбор между Gini и entropy обычно менее важен, чем ограничение глубины и размера листьев.


## 6. Прогноз в листе

Для классификации дерево хранит доли классов среди учебных объектов листа. Самый частый класс становится `predict`, а доли — `predict_proba`.

Эти доли являются оценками вероятностей, но не обязаны быть хорошо откалиброваны, особенно в маленьких листьях. Калибровку проверяют отдельно.

Для регрессии лист обычно предсказывает среднее целевых значений. Разбиения уменьшают квадрат ошибки.


In [ ]:
prices_in_leaf=np.array([8.,10.,12.])
print('Прогноз регрессионного листа:',prices_in_leaf.mean())


In [ ]:
new=pd.DataFrame({'подготовка':[3,7],'сон':[8,5]})
print('Классы:',tree.predict(new))
print('Доли классов в листьях:')
print(tree.predict_proba(new).round(3))
print(export_text(tree,feature_names=list(X.columns)))


In [ ]:
# Пройдём первый новый объект по его пути.
path=tree.decision_path(new.iloc[[0]])
for node in path.indices:
 feature=tree.tree_.feature[node]
 if feature < 0:
  print(f'Узел {node}: лист, прогноз {tree.predict(new.iloc[[0]])[0]}')
 else:
  value=new.iloc[0,feature]; threshold=tree.tree_.threshold[node]
  sign='<=' if value<=threshold else '>'
  print(f'Узел {node}: {X.columns[feature]}={value} {sign} {threshold:.2f}')


## 7. Вероятность и размер листа

Лист из двух объектов может дать вероятность 0 или 1, но такая уверенность ненадёжна. min_samples_leaf заставляет оценивать доли по большему числу примеров.

Поэтому размер листа влияет и на переобучение классов, и на стабильность predict_proba.


## 8. Ступенчатая граница

Каждый вопрос вида $x_j\le t$ делит пространство вертикальной или горизонтальной линией. Несколько вопросов создают прямоугольные области. Поэтому дерево строит нелинейную, но ступенчатую границу.


In [ ]:
x1=np.linspace(0,9,200); x2=np.linspace(3,10,200); xx1,xx2=np.meshgrid(x1,x2)
grid=pd.DataFrame({'подготовка':xx1.ravel(),'сон':xx2.ravel()})
surface=tree.predict_proba(grid)[:,1].reshape(xx1.shape)
plt.contourf(xx1,xx2,surface,levels=[0,.5,1],cmap='RdYlGn',alpha=.35)
plt.scatter(X['подготовка'],X['сон'],c=y,cmap='RdYlGn',edgecolor='black')
plt.xlabel('Подготовка'); plt.ylabel('Сон'); plt.title('Ступенчатая граница дерева'); plt.show()


## 9. Почему граница осевая

В стандартном дереве один вопрос использует один признак. Поэтому каждая линия разбиения параллельна одной из осей.

Наклонную границу дерево приближает лестницей из прямоугольников: чем точнее лестница, тем больше узлов и риск переобучения.


## 10. Почему дерево переобучается

Глубокое дерево может выделить отдельный лист почти для каждого объекта. Train-ошибка станет нулевой, но правила будут описывать шум.

Главные ограничения:

- `max_depth` — глубина;
- `min_samples_split` — минимум объектов для нового разбиения;
- `min_samples_leaf` — минимум объектов в листе.

Их выбирают по validation или кросс-валидации.

**Обрезка после построения (*)** удаляет ветви, польза которых слишком мала. В sklearn её силу задаёт `ccp_alpha`: при росте `ccp_alpha` дерево становится короче.


In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Xm,ym=make_moons(n_samples=350,noise=.3,random_state=42)
Xt,Xv,yt,yv=train_test_split(Xm,ym,test_size=.35,stratify=ym,random_state=42)
for d in [1,3,6,None]:
 m=DecisionTreeClassifier(max_depth=d,random_state=42).fit(Xt,yt)
 print('depth',d,'train',round(accuracy_score(yt,m.predict(Xt)),3),'validation',round(accuracy_score(yv,m.predict(Xv)),3))


In [ ]:
for alpha in [0.0,0.005,0.02,0.08]:
 pruned=DecisionTreeClassifier(ccp_alpha=alpha,random_state=42).fit(Xt,yt)
 print('ccp_alpha',alpha,'листьев',pruned.get_n_leaves(),'глубина',pruned.get_depth())


## 11. Предварительное ограничение и post-pruning

max_depth и min_samples_leaf не дают дереву чрезмерно разрастись — это предварительное ограничение.

ccp_alpha сначала допускает построение ветвей, а затем удаляет недостаточно полезные — post-pruning. Оба пути выбирают по validation.


## 12. Масштаб, категории и пропуски

Дерево сравнивает порядок значений с порогом, поэтому масштабирование обычно не требуется. Строго монотонное преобразование меняет пороги, но сохраняет порядок.

`DecisionTreeClassifier` sklearn не поддерживает категории напрямую: их нужно закодировать без создания ложного порядка. Современные версии sklearn умеют направлять числовые `NaN` при некоторых настройках обычного дерева; поддержка зависит от версии, критерия и режима. Поэтому перед проектом проверяют документацию своей версии и явно фиксируют способ обработки пропусков.


## 13. Числовые пропуски и категории

Поддержка NaN не превращает категорию в число. Категориальные признаки всё равно требуют осмысленного кодирования или алгоритма с нативной поддержкой категорий.

Даже если версия sklearn принимает NaN, полезно понимать, как пропуски будут направляться и сохранится ли это поведение после внедрения.


## 14. Важность признаков

`feature_importances_` суммирует уменьшения impurity, сделанные признаком. Это не доказательство причинности и не гарантирует, что признак полезен на новых данных.

Важность может завышаться у признаков с множеством возможных разбиений и делиться между коррелирующими признаками. Надёжнее дополнять её permutation importance на отдельных данных.


In [ ]:
pd.Series(tree.feature_importances_,index=X.columns).sort_values(ascending=False)


## 15. Permutation importance

Столбец перемешивают на validation, разрушая связь с целью. Падение качества показывает, насколько модель использовала признак на новых данных.

Коррелирующие признаки способны заменять друг друга, поэтому перемешивание одного из них может почти не ухудшить качество.


## 16. Плюсы и ограничения

Плюсы: нелинейность, взаимодействия, понятные правила, отсутствие необходимости масштабировать.

Ограничения: нестабильность, ступенчатые прогнозы, сильное переобучение, слабая экстраполяция.

> **Главная мысль.** Дерево повторяет один шаг: выбирает вопрос, делит данные и снова делает это отдельно в каждой полученной группе. Такое повторение называется рекурсивным; сложность дерева нужно обязательно ограничивать.


## 17. Нестабильность дерева

Небольшое изменение train может поменять первый вопрос и перестроить всё дерево. Это не обязательно снижает среднее качество, но делает правила неустойчивыми.

На этой проблеме будет построено следующее занятие: много различающихся деревьев можно усреднить.
